In [2]:
import bw2data as bd

In [4]:
list(bd.projects)

[Project: default,
 Project: perocillo_cut_off_311,
 Project: cilleruelo_comparison_ei-3.8-cutoff,
 Project: multifunctional_cutoff,
 Project: default_25,
 Project: new_project_eiv2,
 Project: multifunctional_cutoff_bw25,
 Project: BAFU_2.1_zeyao,
 Project: premise_consequential_default,
 Project: pc_dup_bw-25_ecoinvent-3.12-apos_fm-fe,
 Project: pc_dup_bw-25_ecoinvent-3.12-cutoff_fm-fe,
 Project: ecoinvent-3.12-cutoff_edges,
 Project: motorforconf_SR-C_cutoff,
 Project: RA_ei-3.12_onlybw,
 Project: RA_ei-3.12_restored,
 Project: review_example,
 Project: material-composition,
 Project: CERC,
 Project: circularity_sensitivity_analysis,
 Project: brightcon,
 Project: CERC_3.12_consequential,
 Project: pc_dup_bw-25_ecoinvent-3.11-cutoff_fm-fe,
 Project: another_project,
 Project: pc_dup_bw-25_ecoinvent-3.12-consequential_fm-fe,
 Project: X,
 Project: Y,
 Project: libs_circularity_brightcon,
 Project: ecoinvent-3.12-consequential_fm-fe,
 Project: NAME]

In [5]:
bd.projects.set_current("NAME")

In [6]:
bd.databases

Databases dictionary with 3 object(s):
	bafu-2026
	bafu-2026-residual
	ef-3.1-biosphere

# Step 1: analyse aggregated datasets of a brightway project

## functions

In [7]:
from collections import defaultdict
import time
import os
import json
import logging
import bw2data as bd

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def find_aggregated_unit_processes(db_name):
    """
    Find aggregated unit processes in a database.
    Aggregated unit processes are defined as activities with:
    - Exactly one production exchange.
    - No technosphere exchanges.

    Args:
        db_name (str): Name of the database to analyze.

    Returns:
        list: List of activities that meet the criteria.
    """
    db = bd.Database(db_name)
    aggregated_unit_processes = []

    for act in db:
        # Check if the activity has exactly one production exchange
        production_exchanges = list(act.production())
        if len(production_exchanges) != 1:
            continue

        # Check if the activity has no technosphere exchanges
        technosphere_exchanges = list(act.technosphere())
        if len(technosphere_exchanges) > 0:
            continue

        aggregated_unit_processes.append(act)

    logger.info(f"Found {len(aggregated_unit_processes)} aggregated unit processes in database: {db_name}")
    for act in aggregated_unit_processes[:5]:
        logger.info(f"- {act.get('name')} ({act.get('location')}, {act.get('unit')})")
    if len(aggregated_unit_processes) > 5:
        logger.info("...")

    return aggregated_unit_processes

def analyze_all_databases_for_aggregated_unit_processes():
    """
    Analyze all databases for aggregated unit processes.

    Returns:
        dict: A dictionary with database names as keys and lists of aggregated unit processes as values.
    """
    logger.info("Analyzing all databases for aggregated unit processes...")
    all_aggregated_unit_processes = {}
    total = 0
    start = time.time()

    for db_name in bd.databases:
        processes = find_aggregated_unit_processes(db_name)
        all_aggregated_unit_processes[db_name] = processes
        total += len(processes)

    logger.info(f"Total aggregated unit processes found: {total}")
    logger.info(f"Analysis completed in {time.time() - start:.2f} s")

    return all_aggregated_unit_processes

def save_aggregated_unit_processes(results, filename=None):
    """
    Save aggregated unit processes to a JSON file.

    Args:
        results (dict): Dictionary of aggregated unit processes by database.
        filename (str, optional): Name of the JSON file. Defaults to "aggregated_unit_processes.json".
    """
    json_dir = os.path.join("results", "json")
    os.makedirs(json_dir, exist_ok=True)

    if filename is None:
        filename = "aggregated_unit_processes.json"

    filepath = os.path.join(json_dir, filename)

    with open(filepath, "w") as f:
        json.dump(
            {db_name: [act.as_dict() for act in acts]
             for db_name, acts in results.items()},
            f,
            indent=2
        )
    logger.info(f"Aggregated unit processes saved to: {filepath}")

## exe

In [8]:
# Analyze all databases for aggregated unit processes
aggregated_processes = analyze_all_databases_for_aggregated_unit_processes()

# Save the results to a JSON file
save_aggregated_unit_processes(aggregated_processes)

INFO:__main__:Analyzing all databases for aggregated unit processes...
INFO:__main__:Found 0 aggregated unit processes in database: ef-3.1-biosphere
INFO:__main__:Found 0 aggregated unit processes in database: bafu-2026-residual
INFO:__main__:Found 276 aggregated unit processes in database: bafu-2026
INFO:__main__:- Titanium dioxide, chloride process, at plant (RER, kilogram)
INFO:__main__:- Waste Cooking Oil (RER, megajoule)
INFO:__main__:- xx MnO2 powder, pyrometallurgical processing Li-ion batteries, at plant (GLO, kilogram)
INFO:__main__:- xx Al fraction, mechanical treatment, printer, laser, at plant (GLO, kilogram)
INFO:__main__:- Disposal, sheet pile wall, strutted apart, vibrated, Schulhaus Sandgruben Basel (CH, square meter)
INFO:__main__:...
INFO:__main__:Total aggregated unit processes found: 276
INFO:__main__:Analysis completed in 71.42 s
INFO:__main__:Aggregated unit processes saved to: results\json\aggregated_unit_processes.json


# Step 2: analyse data that can be mined

## full db general info

In [12]:
#⚠️⏰
from bw2data import databases, Database

# Initialize a set to store all unique exchange keys
all_exchange_keys = set()
all_upr_keys = set()

db_name = "bafu-2026"
# Iterate through all databases
for db_name in databases:

    db = Database(db_name)
    # Iterate through all activities in the database
    for act in db:
        all_upr_keys.update(act.keys())
        # Iterate through all exchanges in the activity
        for e in act.exchanges():
            # Add all keys from this exchange to the set
            all_exchange_keys.update(e.keys())

# Print all unique keys
print("All possible exchange attributes in your project:")
for key in sorted(all_exchange_keys):
    print(f"- {key}")
print("All possible activity attributes in your project:")
for key in sorted(all_upr_keys):
    print(f"- {key}")


All possible exchange attributes in your project:
- amount
- input
- loc
- name
- negative
- output
- scale
- type
- uncertainty type
- unit
All possible activity attributes in your project:
- CAS number
- categories
- code
- comment
- database
- id
- location
- name
- production amount
- reference product
- type
- unit


## test on a process

In [13]:
def find_processes_by_name(name_part, loc):
    """Find all processes matching a name pattern"""
    matches = []
    for db_name in bd.databases:
        db = bd.Database(db_name)
        for act in db: # type: ignore
            if name_part.lower() in act.get('name', '').lower() and loc.lower() in act.get('location', '').lower():
                matches.append(act)
    return matches
    matches.append(act)


# Find all anaerobic digestion processes
process_name = "Titanium dioxide, chloride process, at plant"
location = "RER"
target_processes = find_processes_by_name(process_name, location)

if not target_processes:
    raise ValueError(f"No processes found matching '{process_name}'")

print(f"Found {len(target_processes)} matching processes:")
for i, proc in enumerate(target_processes):
    print(f"{i+1}. {proc.get('name')} ({proc.get('location')}) - Key: {proc.key}")

# Select the first one (or choose specific one)
target_process = target_processes[0]  # Change index to select different version
print(f"\nSelected process: {target_process.get('name')}")
print(f"Location: {target_process.get('location')}")
print(f"Key: {target_process.key}")
print(f"Database: {target_process['database']}")

Found 1 matching processes:
1. Titanium dioxide, chloride process, at plant (RER) - Key: ('bafu-2026', 'bc92bf5a-94f5-3d3c-865f-278a6ba99440')

Selected process: Titanium dioxide, chloride process, at plant
Location: RER
Key: ('bafu-2026', 'bc92bf5a-94f5-3d3c-865f-278a6ba99440')
Database: bafu-2026


In [14]:
import bw2data as bd

def print_process_metadata(process):
    """
    Print all metadata of a given Brightway2 process.

    Args:
        process: A Brightway2 activity/process object.
    """
    print("\n=== Process Metadata ===")
    for key, value in process.as_dict().items():
        print(f"{key}: {value}")

def print_exchange_metadata(exchange, exchange_type):
    """
    Print metadata for a single exchange.

    Args:
        exchange: A Brightway2 exchange object.
        exchange_type (str): Type of exchange (e.g., "Technosphere", "Biosphere", "Production").
    """
    print(f"\n--- {exchange_type} Exchange ---")
    for key, value in exchange.as_dict().items():
        print(f"{key}: {value}")

def print_all_exchanges_metadata(process):
    """
    Print metadata for all exchanges of a given process.
    Exchanges include: production, technosphere, and biosphere.

    Args:
        process: A Brightway2 activity/process object.
    """
    print("\n=== Exchanges Metadata ===")

    # Production exchanges
    production_exchanges = list(process.production())
    print(f"\nProduction Exchanges ({len(production_exchanges)}):")
    for exc in production_exchanges:
        print_exchange_metadata(exc, "Production")

    # Technosphere exchanges
    technosphere_exchanges = list(process.technosphere())
    print(f"\nTechnosphere Exchanges ({len(technosphere_exchanges)}):")
    for exc in technosphere_exchanges:
        print_exchange_metadata(exc, "Technosphere")

    # Biosphere exchanges
    biosphere_exchanges = list(process.biosphere())
    print(f"\nBiosphere Exchanges ({len(biosphere_exchanges)}):")
    for exc in biosphere_exchanges:
        print_exchange_metadata(exc, "Biosphere")

def print_process_and_exchanges_metadata(process):
    """
    Print metadata for a process and all its exchanges.

    Args:
        process: A Brightway2 activity/process object.
    """
    print_process_metadata(process)
    print_all_exchanges_metadata(process)

In [15]:
# Print metadata for the selected process and its exchanges
print_process_and_exchanges_metadata(target_process)


=== Process Metadata ===
name: Titanium dioxide, chloride process, at plant
reference product: Titanium dioxide, chloride process, at plant
unit: kilogram
location: RER
type: processwithreferenceproduct
production amount: 1.0
comment: BAFU category: chemicals / inorganic. Source: Life Cycle Inventory database of the Swiss Federal Administration, BAFU:2026
database: bafu-2026
code: bc92bf5a-94f5-3d3c-865f-278a6ba99440
id: 360348911878029312

=== Exchanges Metadata ===

Production Exchanges (1):

--- Production Exchange ---
input: ('bafu-2026', 'bc92bf5a-94f5-3d3c-865f-278a6ba99440')
type: production
amount: 1.0
unit: kilogram
name: Titanium dioxide, chloride process, at plant
output: ('bafu-2026', 'bc92bf5a-94f5-3d3c-865f-278a6ba99440')

Technosphere Exchanges (0):

Biosphere Exchanges (1301):

--- Biosphere Exchange ---
input: ('ef-3.1-biosphere', '08a91e70-3ddc-11dd-9bca-0050c2490048')
type: biosphere
amount: 9.6252e-06
unit: kilogram
name: Manganese
output: ('bafu-2026', 'bc92bf5a-9